# P4 — El operador de walk **es** atención

La idea central. Un Transformer actualiza el token `j` con una suma ponderada `h_j = Σ_i α_{ji} v_i` con pesos
**aprendidos** sobre **todos los pares**. Agregar sobre recorridos de longitud `g` tiene la *misma forma* — y
el álgebra de caminos aporta un **soporte sparse multi-salto**: el nodo `i` atiende a `j` solo cuando un
recorrido de longitud `g` los conecta.

**Figuras:** (1) las máscaras de atención sparse por rango, (2) pesos fijos vs aprendidos, (3) la matriz de
atención.

In [ ]:
import torch, numpy as np
from oversquash import viz
from oversquash.data_bottleneck import make_bottleneck_graph
from oversquash.ideal_bridge import walk_operators
from oversquash.attention import WalkAttention
from torch_geometric.utils import softmax as pyg_softmax
data = make_bottleneck_graph(5, 4, 3, torch.Generator().manual_seed(0))
ei, N = data.edge_index.numpy(), data.num_nodes
raw, _ = walk_operators(ei, N, max_length=4)
t = int(data.root_mask.nonzero()[0])

## Figura 1 — el soporte de atención es sparse (y el álgebra lo elige)

Verde = un par `(i,j)` conectado por un recorrido de longitud `g`. La atención estándar llenaría todo el
cuadrado; aquí conservamos solo ~2% en el rango más profundo. Alcance global, costo mínimo.

In [ ]:
viz.plot_mask_grid(raw, N, title='máscaras de alcanzabilidad = el soporte de atención',
                   subtitle_fmt='g={g}: {k}/{N2}')

## Figura 2 — pesos fijos vs aprendidos

Sobre los **mismos** pares alcanzables hacia el objetivo: `walkraw` usa pesos fijos (conteo crudo → todos
iguales, no puede seleccionar); `WalkAttention` aprende pesos query·key (difieren por fuente → puede
seleccionar).

In [ ]:
g = 3
mask = torch.from_numpy((raw[g] > 0).T.astype('float32')).to_sparse().coalesce()
tgt, src = mask.indices()[0], mask.indices()[1]
fixed = raw[g][src.numpy(), tgt.numpy()].astype(float)
torch.manual_seed(0)
layer = WalkAttention(data.x.size(1), 8, n_heads=1)
q = layer.q(data.x).view(N,1,8); k = layer.k(data.x).view(N,1,8)
alpha = pyg_softmax((q[tgt]*k[src]).sum(-1)*layer.scale, tgt, num_nodes=N).detach().numpy().ravel()
into_t = (tgt.numpy() == t)
fixed_n = fixed[into_t] / fixed[into_t].sum()
viz.plot_fixed_vs_learned(src.numpy()[into_t], fixed_n, alpha[into_t],
                          'pesos hacia el objetivo (mismas fuentes)',
                          legend=('fijo (walkraw)', 'aprendido (atención)'))

## Figura 3 — la matriz de atención sobre el soporte

Los pesos de atención aprendidos como matriz (no cero solo en pares alcanzables). Compara con la Figura 1:
mismo soporte sparse, pero ahora cada entrada es un número *aprendido*, no un conteo fijo.

In [ ]:
Amat = np.zeros((N, N))
Amat[tgt.numpy(), src.numpy()] = alpha   # [objetivo, fuente]
viz.plot_multiplicity_heatmap(Amat, 'pesos de atención aprendidos en rango 4')
print('El álgebra dice QUIÉN atiende a quién; la atención aprende CUÁNTO. La prueba en P5.')